#### Q1
Embed the following query:

    How does approximate nearest neighbor search work?

The embedder returns a vector of 384 numbers. What's the first value (v[0])?

- -0.31
- -0.02
- 0.12
- 0.44

In [1]:
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/download.py
!curl -O https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/02-vector-search/embed/embedder.py

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1376  100  1376    0     0   2931      0 --:--:-- --:--:-- --:--:--  2933
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  1520  100  1520    0     0   4646      0 --:--:-- --:--:-- --:--:--  4648


In [2]:
!uv run python download.py

  exists models/Xenova/all-MiniLM-L6-v2/tokenizer.json
  exists models/Xenova/all-MiniLM-L6-v2/model.onnx


In [3]:
from embedder import Embedder
embed = Embedder()

embed.encode("How does approximate nearest neighbor search work?")[0]

np.float64(-0.020582036807885073)

In [4]:
# -0.02

#### Q2
The embedder returns normalized vectors, so the dot product between two of them is their cosine similarity.

Take the page 02-vector-search/lessons/07-sqlitesearch-vector.md, embed its content, and compute the cosine similarity with the query vector from Q1.

What do you get?

- 0.07
- 0.37
- 0.68
- 0.92

In [5]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

documents = [file.parse() for file in reader.read()]
len(documents)

72

In [6]:
v1 = embed.encode("How does approximate nearest neighbor search work?")
v2 = None
for i in documents:
    if i['filename'] == '02-vector-search/lessons/07-sqlitesearch-vector.md':
        print(i)
        v2 = embed.encode(i['content'])

{'content': '# Vector Search with sqlitesearch\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=csxKescwJYM&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous section we used minsearch for vector search.\n\nIt works, but it has three problems:\n\n- It rebuilds the index on every startup\n- It keeps everything in memory\n- It searches by brute force\n\n\nWith text search we never felt these. Indexing was fast because we\ndidn\'t embed anything. With vector search, indexing runs a neural\nnetwork over every document, so it takes a minute on our dataset.\nKeeping everything in memory is fine here, but a larger dataset would\nneed too much space.\n\nThe third problem is brute-force search. For every query we compare the\nquery vector against every single document. With 1,000 documents this is\nfine, probably even faster than anything smarter. But as the dataset\ngrows past 10,000 or so, it gets slow, and we\'ll want an approximate\nmethod instead.\n\nWhat we\'ve done 

In [7]:
len(v1), len(v2)

(384, 384)

In [8]:
v1.dot(v2)

np.float64(0.361070280302606)

In [9]:
# 0.37

#### Q3
Which file does the highest-scoring chunk belong to (its filename)?

- 02-vector-search/lessons/03-embeddings-dataset.md
- 02-vector-search/lessons/06-rag-vector.md
- 02-vector-search/lessons/07-sqlitesearch-vector.md
- 02-vector-search/lessons/09-onnx-embedder.md

In [10]:
from gitsource import chunk_documents
from tqdm.auto import tqdm
import numpy as np

chunks = chunk_documents(documents, size=2000, step=1000)

X = []

for i in tqdm(range(0, len(chunks))):
    batch_vectors = embed.encode_batch([chunks[i]['content']])
    X.extend(batch_vectors)

X = np.array(X)

  0%|          | 0/295 [00:00<?, ?it/s]

In [11]:
len(X), len(chunks)

(295, 295)

In [12]:
scores = X.dot(v1)
chunks[np.argmax(scores)]['filename']

'02-vector-search/lessons/07-sqlitesearch-vector.md'

In [13]:
# 02-vector-search/lessons/07-sqlitesearch-vector.md

#### Q4
We've done vector search by hand, which is good for learning, but it's not what we do in practice. In practice we use libraries.

Let's use VectorSearch from minsearch and run a search for the following query:

    What metric do we use to evaluate a search engine?

Which file is the filename of the first result?

- 02-vector-search/lessons/04-vector-search.md
- 04-evaluation/lessons/05-search-metrics.md
- 04-evaluation/lessons/13-llm-as-judge.md
- 05-monitoring/lessons/04-metrics.md

In [14]:
from minsearch import VectorSearch
v4 = embed.encode("What metric do we use to evaluate a search engine?")

vindex = VectorSearch()
vindex.fit(X, chunks)

results = vindex.search(v4, num_results=5)

In [15]:
results[0]

{'start': 0,
 'content': "# Search Evaluation Metrics\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=TuirMy3Pdbk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we computed relevance lists for search results.\nWe can turn those lists into metrics.\n\n## Hit Rate\n\nHit Rate (also called Recall@k) measures the fraction of queries where\nthe correct document appears anywhere in the results:\n\n```python\nexample = [\n    [1, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 0, 0, 0],\n    [0, 1, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [0, 0, 1, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n    [1, 0, 0, 0, 0],\n]\n```\n\nEach line is one query. If a line contains `1`, search found the\ncorrect document somewhere in the top 5 results. If the line contains\nonly zeros, search did not find the correct document.\n\nIn our setup

In [16]:
# 04-evaluation/lessons/05-search-metrics.md

#### Q5
Vector search matches by meaning, keyword search by exact words.

Let's compare them. Index the same chunks with Index from minsearch. Use content as a text field.

Run both searches for this query:

    How do I store vectors in PostgreSQL?

Take the top 5 results from each method. Which file shows up in the vector results but not in the text results?

- 02-vector-search/lessons/01-intro.md
- 02-vector-search/lessons/02-embeddings.md
- 02-vector-search/lessons/08-pgvector.md
- 03-orchestration/lessons/05-rag.md

In [17]:
v5 = embed.encode("How do I store vectors in PostgreSQL?")
for i in vindex.search(v5, num_results=5):
    print(i['filename'])

02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/08-pgvector.md
02-vector-search/lessons/08-pgvector.md


In [18]:
from minsearch import Index

tindex = Index(
    text_fields=['content'],
)
tindex.fit(chunks)

In [19]:
for i in tindex.search("How do I store vectors in PostgreSQL?", num_results=5):
    print(i['filename'])

02-vector-search/lessons/02-embeddings.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md
03-orchestration/lessons/05-rag.md
02-vector-search/lessons/01-intro.md


In [20]:
# 02-vector-search/lessons/08-pgvector.md

#### Q6
Which file is ranked first after RRF?

- 01-agentic-rag/lessons/01-intro.md
- 01-agentic-rag/lessons/13-function-calling.md
- 01-agentic-rag/lessons/14-agentic-loop.md
- 01-agentic-rag/lessons/16-other-frameworks.md

In [21]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [22]:
q6 = "How do I give the model access to tools?"
v6 = embed.encode(q6)

In [23]:
vector_results = vindex.search(v6, num_results=5)
text_results = tindex.search(q6, num_results=5)

results = rrf([vector_results, text_results])

In [24]:
for i in results:
    print(i['filename'])

01-agentic-rag/lessons/13-function-calling.md
01-agentic-rag/lessons/01-intro.md
01-agentic-rag/lessons/14-agentic-loop.md
04-evaluation/lessons/02-ground-truth.md
01-agentic-rag/lessons/16-other-frameworks.md


In [25]:
# 01-agentic-rag/lessons/13-function-calling.md